# Phase 3 — Baseline Custom CNN training

This notebook builds, trains, and saves a custom Convolutional Neural Network. It is the baseline used later to compare against MobileNetV2.

Before running: choose **Runtime → Change runtime type → T4 GPU**. The first cell will confirm GPU availability.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

PROJECT_ROOT = Path('/content/drive/MyDrive/waste-classification')
SPLIT_DIR = PROJECT_ROOT / 'outputs' / 'phase2_splits'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'phase3_custom_cnn'
MODEL_DIR = PROJECT_ROOT / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CLASSES = ('plastic', 'paper', 'glass', 'metal', 'organic')
CLASS_TO_INDEX = {class_name: index for index, class_name in enumerate(TARGET_CLASSES)}
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
RANDOM_SEED = 42

tf.keras.utils.set_random_seed(RANDOM_SEED)
print('TensorFlow version:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU found. Change the Colab runtime to T4 GPU before training.'

In [ ]:
# Reload the saved split manifests, so this notebook is independent of Phase 2's runtime.
train_df = pd.read_csv(SPLIT_DIR / 'train.csv')
val_df = pd.read_csv(SPLIT_DIR / 'validation.csv')
test_df = pd.read_csv(SPLIT_DIR / 'test.csv')

with open(SPLIT_DIR / 'configuration.json') as file:
    saved_config = json.load(file)

class_weights = {int(index): weight for index, weight in saved_config['class_weights'].items()}
print(f'Train: {len(train_df):,} | Validation: {len(val_df):,} | Test: {len(test_df):,}')
print('Class weights:', class_weights)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_resize_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    return tf.cast(image, tf.float32), label

def make_dataset(dataframe, training=False):
    paths = dataframe['path'].astype(str).to_numpy()
    labels = dataframe['label'].map(CLASS_TO_INDEX).to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(dataframe), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    return (dataset
        .map(load_and_resize_image, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE))

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

In [ ]:
# Augmentation is active only during training. Validation and test images remain untouched.
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name='training_augmentation')

custom_cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
    data_augmentation,
    tf.keras.layers.Rescaling(1.0 / 255),

    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.40),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(len(TARGET_CLASSES), activation='softmax', name='waste_class'),
], name='custom_cnn_baseline')

custom_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)

custom_cnn.summary()

In [ ]:
MODEL_PATH = MODEL_DIR / 'custom_cnn_best.keras'
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = custom_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

pd.DataFrame(history.history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)

In [ ]:
# Plot learning curves for the presentation.
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_df['accuracy'], label='Training')
axes[0].plot(history_df['val_accuracy'], label='Validation')
axes[0].set_title('Custom CNN accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history_df['loss'], label='Training')
axes[1].plot(history_df['val_loss'], label='Validation')
axes[1].set_title('Custom CNN loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

fig.suptitle('Custom CNN training history')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Load the checkpoint selected by validation accuracy, then evaluate it once
# on the untouched test split.
best_custom_cnn = tf.keras.models.load_model(MODEL_PATH)
test_loss, test_accuracy = best_custom_cnn.evaluate(test_ds, verbose=1)
results = {
    'test_loss': float(test_loss),
    'test_accuracy': float(test_accuracy),
    'best_model_path': str(MODEL_PATH),
}

with open(OUTPUT_DIR / 'test_results.json', 'w') as file:
    json.dump(results, file, indent=2)

print(f'Custom CNN test accuracy: {test_accuracy:.2%}')
print('Best model saved to:', MODEL_PATH)